# Goal Setting and Monitoring

The Goal Setting and Monitoring pattern enables agents to pursue a defined objective through an iterative **generate → evaluate → refine** loop. The agent doesn't stop at the first output — it measures success against explicit criteria and continues until goals are met or a maximum iteration count is reached.

## Implementation with Flyte v2

This notebook reimplements the LangChain + OpenAI iterative code-generation agent from Chapter 11 using **Flyte v2 primitives only**.

#### LangChain vs Flyte v2 — Key Differences

| Aspect | LangChain | Flyte v2 |
|--------|-----------|----------|
| **LLM client** | `ChatOpenAI` via LangChain wrapper | Direct `AsyncAnthropic` client |
| **State** | In-process variables (lost on failure) | Typed `GoalEvaluation` dataclass — durable |
| **Checkpointing** | None (restart from scratch on failure) | `@flyte.trace` per LLM call — resumes from checkpoint |
| **Live progress** | `print()` statements | `flyte.report` HTML tab updated each iteration |
| **Retries** | Manual `try/except` wrapping | `retries=3` on `@env.task` |
| **Secrets** | `.env` / `dotenv.load_dotenv()` | `flyte.Secret` injected by cluster |

### 1. Install dependencies

In [ ]:
!uv pip install 'flyte[tui]' anthropic

### 2. Store your API key

In [ ]:
!flyte create secret ANTHROPIC_API_KEY --value sk-ant-...

### 3. Import dependencies and configure the TaskEnvironment

In [ ]:
from __future__ import annotations

import os
from dataclasses import dataclass, field
from datetime import timedelta

import anthropic
import flyte
import flyte.report

flyte.init_from_config()

_image = (
    flyte.Image.from_debian_base(name="goal-agent", python_version=(3, 12))
    .with_pip_packages("anthropic>=0.40.0")
)

goal_env = flyte.TaskEnvironment(
    name="goal_agent",
    image=_image,
    resources=flyte.Resources(cpu="1", memory="2Gi"),
    secrets=[
        flyte.Secret(key="ANTHROPIC_API_KEY", as_env_var="ANTHROPIC_API_KEY"),
    ],
    reusable=flyte.ReusePolicy(
        replicas=(1, 4),
        concurrency=4,
        scaledown_ttl=timedelta(minutes=5),
        idle_ttl=timedelta(minutes=10),
    ),
)

### 4. Define data models

The LangChain example uses in-process variables (`previous_code`, `feedback`) to track state across iterations. In Flyte v2, each iteration's state is captured in typed dataclasses — serializable, durable, and inspectable in the UI.

`GoalEvaluation` replaces the `goals_met()` function's boolean return — it carries both the verdict and the reasoning, enabling richer monitoring.

In [ ]:
@dataclass
class GoalEvaluation:
    """Structured output from the evaluator LLM call."""
    goals_met: bool
    feedback: str
    iteration: int


@dataclass
class GoalAgentResult:
    """Final output of the goal-setting workflow."""
    final_code: str
    iterations_used: int
    converged: bool
    evaluations: list[GoalEvaluation] = field(default_factory=list)

### 5. Define traced LLM helpers

`@flyte.trace` turns each LLM call into a checkpoint. On pod failure mid-loop, the task resumes from the last successful checkpoint rather than restarting from iteration 0.

The **generator** produces code; the **evaluator** judges whether the goals are met. Separating these calls (as in the LangChain example) prevents the self-serving rationalization problem: the evaluator sees only the code and goals, not the generator's reasoning context.

In [ ]:
GENERATOR_SYSTEM = """\
You are an expert Python developer.
Generate clean, idiomatic Python code that satisfies the stated use case and goals.
Return ONLY the raw Python source — no prose, no markdown fences."""

EVALUATOR_SYSTEM = """\
You are a strict code reviewer. Evaluate the provided Python code against the stated goals.
Respond with exactly TWO lines:
Line 1: PASS or FAIL
Line 2: One sentence of specific, actionable feedback (even for PASS)."""


@flyte.trace
async def _generate(use_case: str, goals: list[str], previous_code: str, feedback: str) -> str:
    client = anthropic.AsyncAnthropic(api_key=os.environ["ANTHROPIC_API_KEY"])
    content = f"Use case: {use_case}\n\nGoals:\n" + "\n".join(f"- {g}" for g in goals)
    if previous_code:
        content += f"\n\nPrevious code (revise this):\n{previous_code}"
    if feedback:
        content += f"\n\nFeedback to address:\n{feedback}"
    response = await client.messages.create(
        model="claude-haiku-4-5-20251001",
        max_tokens=2048,
        system=GENERATOR_SYSTEM,
        messages=[{"role": "user", "content": content}],
    )
    return response.content[0].text.strip()


@flyte.trace
async def _evaluate(use_case: str, goals: list[str], code: str, iteration: int) -> GoalEvaluation:
    client = anthropic.AsyncAnthropic(api_key=os.environ["ANTHROPIC_API_KEY"])
    response = await client.messages.create(
        model="claude-haiku-4-5-20251001",
        max_tokens=512,
        system=EVALUATOR_SYSTEM,
        messages=[{
            "role": "user",
            "content": (
                f"Use case: {use_case}\n\nGoals:\n" + "\n".join(f"- {g}" for g in goals)
                + f"\n\nCode to evaluate:\n{code}"
            ),
        }],
    )
    lines = response.content[0].text.strip().splitlines()
    verdict = lines[0].strip().upper() if lines else "FAIL"
    feedback = lines[1].strip() if len(lines) > 1 else "No feedback provided."
    return GoalEvaluation(
        goals_met=(verdict == "PASS"),
        feedback=feedback,
        iteration=iteration,
    )

### 6. Define the goal-setting task

The `goal_agent` task orchestrates the generate → evaluate → refine loop. Key decisions:

- **`cache=flyte.Cache(behavior="disable")`** — code generation is non-deterministic; caching would return stale results.
- **`report=True`** — live HTML report updated each iteration, visible in the Flyte UI.
- **`retries=3`** — transient API failures are retried automatically; the `@flyte.trace` checkpoints prevent re-running completed iterations.

In [ ]:
def _html_escape(text: str) -> str:
    return text.replace("&", "&amp;").replace("<", "&lt;").replace(">", "&gt;")


def _render_report(sections: list[str], completed: int, status: str) -> str:
    return f"""
<!DOCTYPE html><html><head><style>
  body {{ font-family: -apple-system, monospace; padding: 1.5em; max-width: 900px; margin: auto; }}
  h1   {{ border-bottom: 2px solid #333; padding-bottom: .4em; }}
  pre  {{ white-space: pre-wrap; font-size: .85em; background:#f4f4f4; padding:1em; border-radius:4px; }}
</style></head><body>
  <h1>Goal Agent — Live Execution Report</h1>
  <p>Iterations: <strong>{completed}</strong> &nbsp;|&nbsp; Status: <strong>{status}</strong></p>
  {chr(10).join(sections)}
</body></html>
"""


@goal_env.task(
    retries=3,
    timeout=timedelta(minutes=15),
    cache=flyte.Cache(behavior="disable"),
    report=True,
)
async def goal_agent(
    use_case: str,
    goals: list[str],
    max_iterations: int = 5,
) -> GoalAgentResult:
    """
    Goal-setting agent: generate code, evaluate against goals, refine until PASS or max iterations.

    Memory flow:
      1. Generate initial code from use_case + goals.
      2. Evaluate code against goals → GoalEvaluation.
      3. If PASS, stop. If FAIL, pass feedback to generator and repeat.
    """
    current_code = ""
    current_feedback = ""
    evaluations: list[GoalEvaluation] = []
    converged = False
    report_sections: list[str] = []

    for i in range(max_iterations):
        label = f"Iteration {i + 1} / {max_iterations}"

        current_code = await _generate(
            use_case=use_case,
            goals=goals,
            previous_code=current_code,
            feedback=current_feedback,
        )

        evaluation = await _evaluate(
            use_case=use_case,
            goals=goals,
            code=current_code,
            iteration=i + 1,
        )
        evaluations.append(evaluation)
        converged = evaluation.goals_met

        color = "green" if converged else "orange"
        status_label = "GOALS MET" if converged else "NEEDS WORK"
        report_sections.append(
            f"<section><h2>{label} — <span style='color:{color}'>{status_label}</span></h2>"
            f"<h3>Generated Code</h3><pre><code>{_html_escape(current_code)}</code></pre>"
            f"<h3>Evaluation</h3><pre>{_html_escape(evaluation.feedback)}</pre>"
            f"</section><hr/>"
        )
        await flyte.report.replace.aio(_render_report(report_sections, i + 1, status_label))
        await flyte.report.flush.aio()

        if converged:
            break

        current_feedback = evaluation.feedback

    return GoalAgentResult(
        final_code=current_code,
        iterations_used=len(evaluations),
        converged=converged,
        evaluations=evaluations,
    )

### 7. Run locally

In [ ]:
USE_CASE = "Write a Python function that finds the BinaryGap of a positive integer."

GOALS = [
    "Functionally correct for all valid positive integers",
    "Handles edge cases: no gaps (return 0), single bit",
    "Includes a clear docstring with examples",
    "Raises ValueError for non-positive inputs",
    "Simple and readable — no unnecessary complexity",
]

run = flyte.run(goal_agent, use_case=USE_CASE, goals=GOALS, max_iterations=4)
run.wait()
result: GoalAgentResult = run.outputs()[0]

print(f"Converged: {result.converged}")
print(f"Iterations used: {result.iterations_used}")
print("\n" + "=" * 60)
print(result.final_code)

### Running remotely

Remote execution adds the live `report` tab to the Flyte UI, showing each iteration's generated code and evaluation in real time.

In [ ]:
run = flyte.run(goal_agent, use_case=USE_CASE, goals=GOALS, max_iterations=4)
run.wait()
result = run.outputs()[0]
print(f"Converged: {result.converged}, Iterations: {result.iterations_used}")
print(result.final_code)